# Sales Performance Analysis
**Goal:** Explore sales data to identify trends, top performers, and profitability drivers.

**Steps:**
1. Load and inspect data
2. Univariate analysis (distributions)
3. Revenue trends over time
4. Category and region breakdown
5. Discount vs. profit relationship
6. Key takeaways

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path

# Plot style
sns.set_theme(style='darkgrid', palette='Blues_d')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.titlesize'] = 14

DATA = Path('../data/cleaned/cleaned_data.csv')
df = pd.read_csv(DATA, parse_dates=['order_date', 'ship_date'])
print(f'Loaded {len(df):,} rows × {len(df.columns)} columns')
df.head()

In [ ]:
# --- Basic stats ---
print('Shape:', df.shape)
print('\nNull counts:')
print(df.isnull().sum()[df.isnull().sum() > 0])
print('\nData types:')
print(df.dtypes)
df.describe()

In [ ]:
# --- Monthly revenue trend ---
monthly = (
    df.groupby(df['order_date'].dt.to_period('M'))
    [['sales', 'profit']]
    .sum()
    .reset_index()
)
monthly['order_date'] = monthly['order_date'].astype(str)

fig, ax = plt.subplots()
ax.plot(monthly['order_date'], monthly['sales'],  label='Revenue', linewidth=2)
ax.plot(monthly['order_date'], monthly['profit'], label='Profit',  linewidth=2, linestyle='--')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.xticks(rotation=45, ha='right', fontsize=8)
ax.set_title('Monthly Revenue & Profit Trend')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Revenue by category ---
cat = df.groupby('category')[['sales', 'profit']].sum().sort_values('sales', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cat['sales'].plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='none')
axes[0].set_title('Revenue by Category')
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[0].set_xlabel('')

cat['profit'].plot(kind='bar', ax=axes[1], color='teal', edgecolor='none')
axes[1].set_title('Profit by Category')
axes[1].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[1].set_xlabel('')

plt.tight_layout()
plt.show()
print(cat)

In [ ]:
# --- Regional performance ---
regional = df.groupby('region')[['sales', 'profit']].sum().sort_values('sales', ascending=False)

regional.plot(kind='bar', figsize=(10, 5))
plt.title('Revenue & Profit by Region')
plt.xlabel('')
plt.xticks(rotation=0)
plt.gca().yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.show()
print(regional)

In [ ]:
# --- Discount vs. Profit scatter ---
fig, ax = plt.subplots(figsize=(10, 5))
scatter = ax.scatter(
    df['discount'], df['profit'],
    alpha=0.3, c=df['sales'], cmap='Blues', edgecolors='none', s=20
)
plt.colorbar(scatter, ax=ax, label='Sale Amount')
ax.axhline(0, color='red', linewidth=1, linestyle='--', label='Break-even')
ax.set_xlabel('Discount Rate')
ax.set_ylabel('Profit')
ax.set_title('Discount vs. Profit (each dot = one order)')
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Top 10 products by revenue ---
top_products = (
    df.groupby('product_name')['sales']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

top_products.plot(kind='barh', figsize=(10, 6), color='steelblue')
plt.title('Top 10 Products by Revenue')
plt.xlabel('Total Revenue')
plt.gca().invert_yaxis()
plt.gca().xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.show()

## Key Takeaways

1. **Seasonality:** Q4 drives the strongest sales — plan inventory and marketing accordingly.
2. **Category profitability:** Technology leads; Furniture lags — consider pricing review.
3. **Discount danger zone:** Orders with >20% discount almost always produce negative profit.
4. **Regional gaps:** West region outperforms; investigate underperforming states in the South.

**Next step:** Build the Tableau/Power BI dashboard using these findings as the design brief.